# Multi-seed only — Goodreads Poetry

Picks up after a disconnect. `10_multiseed_full.py` rebuilds the splits, experts and
every model itself from the normalised interactions, so the single-seed pipeline does
**not** need to run again — this skips straight to the seeds and saves about an hour.

Same contract as before: **Run all**, answer the two prompts in the first cells, then
leave it. Results auto-save to Drive every 3 minutes, and completed seeds are restored
from your previous run so nothing already finished is repeated.

Each seed is roughly 40–60 minutes. Pick a T4 GPU runtime — Model 7's BiLSTM is ~18
minutes of each seed on CPU.


## 1. Code and dependencies


In [ ]:
%cd /content
!git clone -q https://github.com/Ar555Rathod/adaptive-trust-gate.git 2>/dev/null || git -C adaptive-trust-gate pull -q origin master
%cd /content/adaptive-trust-gate
!git log --oneline -1


In [ ]:
!pip install -q -r requirements.txt
import surprise; print('scikit-surprise OK:', surprise.__version__)


## 2. Mount Drive — the only prompt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Auto-save thread


In [ ]:
import shutil, subprocess, threading, time
from pathlib import Path

SRC = Path('/content/adaptive-trust-gate/results')
RUN_ID = time.strftime('run_%Y%m%d_%H%M')
DST = Path('/content/drive/MyDrive/atg_results') / RUN_ID
DST.mkdir(parents=True, exist_ok=True)

HAVE_RSYNC = shutil.which('rsync') is not None
sync_state = {'count': 0, 'last': None, 'error': None}

def sync_once():
    if not SRC.exists() or not any(SRC.rglob('*')):
        return False
    if HAVE_RSYNC:
        subprocess.run(['rsync', '-a', f'{SRC}/', f'{DST}/'], check=False, capture_output=True)
    else:
        shutil.copytree(SRC, DST, dirs_exist_ok=True)
    return True

def sync_loop(every=180):
    while True:
        try:
            if sync_once():
                sync_state['count'] += 1
                sync_state['last'] = time.strftime('%H:%M:%S')
        except Exception as e:
            sync_state['error'] = repr(e)
        time.sleep(every)

threading.Thread(target=sync_loop, daemon=True).start()
print(f'auto-save ON  ->  {DST}   (additive, existing folders untouched)')


## 4. Dataset and normalisation

Only `normalize.py` is needed — the multi-seed script builds its own splits per seed.


In [ ]:
BASE = 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre'
!mkdir -p data/raw/goodreads
!wget -q -O data/raw/goodreads/goodreads_books_poetry.json.gz {BASE}/goodreads_books_poetry.json.gz
!wget -q -O data/raw/goodreads/goodreads_interactions_poetry.json.gz {BASE}/goodreads_interactions_poetry.json.gz
!ls -lh data/raw/goodreads


In [ ]:
%env ATG_DATASET=goodreads
%env ATG_GOODREADS_GENRE=poetry
%env PYTHONPATH=src


In [ ]:
!python src/atg/data/normalize.py


## 5. Restore what the last run produced

Brings back the previous run's metrics — including the multi-seed checkpoint, so any
seed that already finished is skipped — and keeps the single-seed results alongside the
new ones so the final Drive folder holds everything.


In [ ]:
from pathlib import Path
import shutil, json as _json

root = Path('/content/drive/MyDrive/atg_results')
dest = Path('/content/adaptive-trust-gate/results/goodreads_poetry/metrics')
dest.mkdir(parents=True, exist_ok=True)

runs = sorted((d for d in root.glob('run_*') if d.name != RUN_ID),
              key=lambda d: d.stat().st_mtime, reverse=True)

src_metrics = next((r / 'goodreads_poetry' / 'metrics' for r in runs
                    if (r / 'goodreads_poetry' / 'metrics').is_dir()), None)

if src_metrics:
    shutil.copytree(src_metrics, dest, dirs_exist_ok=True)
    print(f'restored metrics from {src_metrics.parent.parent.name}')
    ck = dest / 'multiseed_full_comparison.json'
    if ck.exists():
        print('  checkpoint found -- seeds already done:', _json.load(open(ck)).get('seeds', []))
    else:
        print('  no multi-seed checkpoint -- the seed run will start from scratch')
else:
    print('no previous run found in Drive -- starting from scratch')


## 6. The seed run

Each seed refits all 11 models and checkpoints on completion. The loop re-invokes the
script on failure; because completed seeds are skipped, a retry resumes rather than
restarting.


In [ ]:
import subprocess, os, time

env = dict(os.environ, ATG_SEEDS='42,1,2')

for attempt in range(1, 6):
    print(f'--- attempt {attempt}  ({time.strftime("%H:%M:%S")}) ---', flush=True)
    r = subprocess.run(['python', '-u', 'scripts/10_multiseed_full.py'], env=env)
    if r.returncode == 0:
        print('multiseed finished cleanly')
        break
    print(f'exit {r.returncode} -- retrying from the last checkpoint')
    sync_once()
else:
    print('did not finish after 5 attempts -- whatever completed is checkpointed and saved')


## 7. Save and check


In [ ]:
print('final sync:', 'done' if sync_once() else 'nothing to copy')
print(f'syncs: {sync_state["count"]}   last: {sync_state["last"]}   error: {sync_state["error"]}')
print(f'saved to: {DST}')


In [ ]:
import json
p = f'{DST}/goodreads_poetry/metrics/multiseed_full_comparison.json'
try:
    d = json.load(open(p))
    print('seeds completed:', d['seeds'], '\n')
    rows = sorted(d['aggregate'].items(), key=lambda kv: kv[1]['overall']['rmse_mean'])
    print(f"{'model':22s} {'overall RMSE':>20s}")
    for name, agg in rows:
        a = agg['overall']
        print(f"{name:22s} {a['rmse_mean']:.4f} +/- {a['rmse_std']:.4f}")
except FileNotFoundError:
    print('not found -- the seed run did not complete:', p)
